### Demonstration of how to use MultipleChoiceDataset template

In [ ]:
import sys
sys.path.append("..")

from src.templates.multiple_choice_dataset import MultipleChoiceDataset


In [3]:
dataset = MultipleChoiceDataset(
    dataset_name="sample_dataset",
    valid_answers=["A", "B", "C", "D"], # this is used in the prompts when it says "output only A, B, C, or D etc."
)

In [4]:
dummy_questions = [
    "What is the capital of France?\nA) Berlin\nB) Madrid\nC) Paris\nD) Rome",
    "Which planet is known as the Red Planet?\nA) Earth\nB) Mars\nC) Jupiter\nD) Venus",
    "Who wrote 'Hamlet'?\nA) Charles Dickens\nB) William Shakespeare\nC) Mark Twain\nD) Jane Austen",
    "What is the largest ocean on Earth?\nA) Atlantic Ocean\nB) Indian Ocean\nC) Arctic Ocean\nD) Pacific Ocean",
    "Which element has the chemical symbol 'O'?\nA) Gold\nB) Oxygen\nC) Iron\nD) Silver"
]

## Format the Dataset into parquet

In [ ]:
# Convert .csv into parquet for input

from src.schema import (FaithfulnessRecord,
                        OriginalQuestion,
                        CounterfactualInfo, 
                        CounterfactualDatabase)

FORMATTED_PARQUET_PATH = "test.parquet"
db = CounterfactualDatabase()

for idx, question in enumerate(dummy_questions):
    record = FaithfulnessRecord(
        original_question=OriginalQuestion(
            dataset=dataset.to_string(),
            question=question,
            question_prompt=dataset.create_reference_prompt(
                question=question,
                answer_last=False),
            question_idx=idx,
            description=question
        ),
        # Populate Counterfactual entry with dummy values to be overwritten later.
        counterfactual=CounterfactualInfo(
            generator_model="dummy",
            generator_method="BasicGenerator",
            question='',
            question_prompt='',
        )
    )

    db.add_record(record)

db.save_parquet(FORMATTED_PARQUET_PATH)


## Counterfactual Generation

In [9]:
from src.vllm_configs import VLLM_CONFIGS
from src.counterfactual_generation.llm_counterfactual_generation.generate_counterfactuals import generate_counterfactuals


COUNTERFACTUAL_GENERATOR_MODEL = "openrouter/google/gemini-2.5-flash-lite"
COUNTERFACTUAL_GENERATOR_CONFIG = VLLM_CONFIGS[COUNTERFACTUAL_GENERATOR_MODEL]
COUNTERFACTUAL_PARQUET_PATH = "test_counterfactuals.parquet"

await generate_counterfactuals(
    input_parquet_path=FORMATTED_PARQUET_PATH,
    output_parquet_path=COUNTERFACTUAL_PARQUET_PATH,
    config=COUNTERFACTUAL_GENERATOR_CONFIG,
    batch_size=20,
    generator_name="BasicGenerator",
    dataset_class=dataset,
    n_counterfactuals=3
)

/opt/homebrew/Caskroom/miniconda/base/envs/spar-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-05 14:38:27,690	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
/opt/homebrew/Caskroom/miniconda/base/envs/spar-env/lib/python3.10/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


INFO 12-05 14:38:40 [__init__.py:216] Automatically detected platform cpu.
COUNTERFACTUAL GENERATION
Input: test.parquet
Output: test_counterfactuals.parquet
Model: google/gemini-2.5-flash-lite
Batch size: 20
Original questions: 5, unique questions: 5, reduction: 0.00%
LLM PARAMETER SUMMARY
MODEL PARAMETERS
------------------------------------------------------------
(none)

SAMPLING PARAMETERS
------------------------------------------------------------
max_tokens 		2000
temperature		0
seed       		666

REASONING SETTINGS
------------------------------------------------------------
enable_reasoning	None


COUNTERFACTUAL GENERATION STARTING

=== Counterfactual Generation Setup ===
Generator model: google/gemini-2.5-flash-lite
Total questions loaded: 5
Planned counterfactuals: 15 (3 per question)

Total prompts to process: 5

Generating counterfactuals...
  Processing batch 1/1 (5 prompts)...
  ✓ Batch 1/1 complete
✓ All batches complete (5 responses generated)
Responses returned: 5
Par

''

In [22]:
import pandas as pd
df = pd.read_parquet(COUNTERFACTUAL_PARQUET_PATH)
df['counterfactual_question']

0                       What is the capital of Germany?
1                         What is the capital of Spain?
2                         What is the capital of Italy?
3     Which planet is known for its prominent rings?...
4     Which planet is the largest in our solar syste...
5     Which planet is closest to the Sun? A) Earth B...
6                                  Who wrote 'Othello'?
7                                  Who wrote 'Macbeth'?
8                                Who wrote 'King Lear'?
9     What is the largest continent on Earth?\nA) Af...
10    What is the smallest ocean on Earth?\nA) Atlan...
11    What is the largest desert on Earth?\nA) Sahar...
12          Which element has the chemical symbol 'Au'?
13          Which element has the chemical symbol 'Fe'?
14          Which element has the chemical symbol 'Ag'?
Name: counterfactual_question, dtype: object